# Defense Comparison — Membership Inference Attack Results

This notebook loads the per-sample MIA feature CSVs from each defense configuration,
runs both threshold and shadow (learned) attacks, and produces a unified comparison
table and plots.

**Defenses compared:**
- No defense (baseline v2)
- Early stopping (gap threshold = 0.08)
- Knowledge distillation (τ=5, α=0.7)
- Confidence masking (Laplace noise, scale=0.2)
- Regularised (L2 + label smoothing + dropout) — if available
- DP-SGD at ε = 1, 5, 10 — if available

**Run from project root.** All paths are relative.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, accuracy_score

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

## 1. Load all available defense CSVs

In [ ]:
LOGS = 'outputs/logs'

# Map: display label -> csv filename
ALL_CONFIGS = {
    'No defense':    'per_sample_mia_v2.csv',
    'Early stop':    'per_sample_mia_early_stop.csv',
    'Distillation':  'per_sample_mia_distillation.csv',
    'Conf masking':  'per_sample_mia_conf_masking.csv',
    'Regularised':   'per_sample_mia_regularised.csv',
    'DP  ε=10':      'per_sample_mia_dp_eps10.csv',
    'DP  ε=5':       'per_sample_mia_dp_eps5.csv',
    'DP  ε=1':       'per_sample_mia_dp_eps1.csv',
}

# Only load configs whose CSV exists
configs = {}
for label, fname in ALL_CONFIGS.items():
    path = os.path.join(LOGS, fname)
    if os.path.exists(path):
        configs[label] = pd.read_csv(path)
        print(f'  ✓ {label:<16} ({len(configs[label]):,} samples)')
    else:
        print(f'  ✗ {label:<16} — not found, skipping')

print(f'\nLoaded {len(configs)} configurations.')

## 2. Helper functions

In [ ]:
def prepare_features(df):
    """Prepare feature matrix for the shadow attack classifier."""
    feat_cols = ['conf_true', 'conf_max', 'loss', 'entropy',
                 'logit_true', 'logit_gap', 'm_entropy']
    x = df[feat_cols].copy()
    x['loss'] = -x['loss']
    x['entropy'] = -x['entropy']
    y = (df['split_name'] == 'member').astype(int).values
    return x.values.astype(np.float64), y


def tpr_at_fpr(fpr_arr, tpr_arr, max_fpr):
    mask = fpr_arr <= max_fpr
    return float(tpr_arr[mask].max()) if mask.any() else 0.0


def evaluate_config(df, attack_clf=None):
    """Run threshold + optional shadow attack on one config."""
    y_true = (df['split_name'] == 'member').astype(int).values

    # Threshold attack — logit_gap
    fpr_t, tpr_t, _ = roc_curve(y_true, df['logit_gap'].values, pos_label=1)
    auc_thresh = auc(fpr_t, tpr_t)
    tpr1_thresh = tpr_at_fpr(fpr_t, tpr_t, 0.01)

    # Shadow attack (if classifier provided)
    auc_shadow = tpr1_shadow = advantage = np.nan
    fpr_s = tpr_s = None
    if attack_clf is not None:
        x_feat, _ = prepare_features(df)
        score = attack_clf.predict_proba(x_feat)[:, 1]
        fpr_s, tpr_s, thresholds = roc_curve(y_true, score, pos_label=1)
        auc_shadow = auc(fpr_s, tpr_s)
        tpr1_shadow = tpr_at_fpr(fpr_s, tpr_s, 0.01)
        best_acc = max(accuracy_score(y_true, (score >= t).astype(int))
                       for t in thresholds)
        advantage = 2 * (best_acc - 0.5)

    mem_acc = df[df.split_name == 'member']['correct'].mean()
    non_acc = df[df.split_name == 'nonmember']['correct'].mean()

    return dict(
        auc_thresh=auc_thresh, tpr1_thresh=tpr1_thresh,
        auc_shadow=auc_shadow, tpr1_shadow=tpr1_shadow,
        advantage=advantage,
        mem_acc=mem_acc, non_acc=non_acc,
        overfit_gap=mem_acc - non_acc,
        fpr_s=fpr_s, tpr_s=tpr_s,
        fpr_t=fpr_t, tpr_t=tpr_t,
    )

## 3. Train shadow attack classifier (once)

In [ ]:
SHADOW_CSV = 'outputs/reports/attack_shadow_v2/shadow_features_v2.csv'

attack_clf = None
if os.path.exists(SHADOW_CSV):
    shadow_df = pd.read_csv(SHADOW_CSV)
    x_shd, y_shd = prepare_features(shadow_df)
    attack_clf = LogisticRegression(max_iter=2000, solver='lbfgs', class_weight='balanced')
    attack_clf.fit(x_shd, y_shd)
    print(f'Shadow attack classifier trained on {len(shadow_df):,} samples.')
else:
    print('Shadow features not found — will only run threshold attacks.')
    print('Run models/attacks/attack-shadow.py first for learned attack results.')

## 4. Evaluate all defenses

In [ ]:
results = {}
for label, df in configs.items():
    results[label] = evaluate_config(df, attack_clf)

# Build summary table
rows = []
for label, r in results.items():
    rows.append({
        'Defense': label,
        'Holdout acc': f"{r['non_acc']:.4f}",
        'Member acc': f"{r['mem_acc']:.4f}",
        'Overfit gap': f"{r['overfit_gap']:.4f}",
        'Thresh AUC': f"{r['auc_thresh']:.4f}",
        'Shadow AUC': f"{r['auc_shadow']:.4f}" if not np.isnan(r['auc_shadow']) else '—',
        'TPR@1% (thresh)': f"{r['tpr1_thresh']:.4f}",
        'TPR@1% (shadow)': f"{r['tpr1_shadow']:.4f}" if not np.isnan(r['tpr1_shadow']) else '—',
        'Advantage': f"{r['advantage']:.4f}" if not np.isnan(r['advantage']) else '—',
    })

summary_df = pd.DataFrame(rows)
summary_df

## 5. ROC curves — all defenses

In [ ]:
PALETTE = {
    'No defense':   '#2C2C2A',
    'Early stop':   '#7F77DD',
    'Distillation': '#E07BAA',
    'Conf masking': '#4ECDC4',
    'Regularised':  '#1D9E75',
    'DP  ε=10':     '#185FA5',
    'DP  ε=5':      '#BA7517',
    'DP  ε=1':      '#D85A30',
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Threshold attack ROC — all defenses', fontsize=13)

for ax_i, ax in enumerate(axes):
    xlim = (0, 1.0) if ax_i == 0 else (0, 0.05)
    ylim = (0, 1.05) if ax_i == 0 else (0, 0.60)

    for label, r in results.items():
        color = PALETTE.get(label, '#888888')
        ls = '--' if label == 'No defense' else '-'
        ax.plot(r['fpr_t'], r['tpr_t'], color=color, lw=2, ls=ls,
                label=f"{label}  (AUC={r['auc_thresh']:.3f})")

    ax.plot([0,1],[0,1], 'k:', lw=1, alpha=0.3)
    ax.axvline(0.01, color='gray', lw=0.7, ls=':', alpha=0.5)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('Full ROC' if ax_i == 0 else 'Zoom: FPR 0–5%', fontsize=11)
    ax.spines[['top','right']].set_visible(False)
    if ax_i == 0:
        ax.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.show()

## 6. Shadow attack ROC (if available)

In [ ]:
if attack_clf is not None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle('Shadow (learned) attack ROC — all defenses', fontsize=13)

    for ax_i, ax in enumerate(axes):
        xlim = (0, 1.0) if ax_i == 0 else (0, 0.05)
        ylim = (0, 1.05) if ax_i == 0 else (0, 0.60)

        for label, r in results.items():
            if r['fpr_s'] is None:
                continue
            color = PALETTE.get(label, '#888888')
            ls = '--' if label == 'No defense' else '-'
            ax.plot(r['fpr_s'], r['tpr_s'], color=color, lw=2, ls=ls,
                    label=f"{label}  (AUC={r['auc_shadow']:.3f})")

        ax.plot([0,1],[0,1], 'k:', lw=1, alpha=0.3)
        ax.axvline(0.01, color='gray', lw=0.7, ls=':', alpha=0.5)
        ax.set_xlim(*xlim); ax.set_ylim(*ylim)
        ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
        ax.set_title('Full ROC' if ax_i == 0 else 'Zoom: FPR 0–5%', fontsize=11)
        ax.spines[['top','right']].set_visible(False)
        if ax_i == 0:
            ax.legend(fontsize=8, loc='lower right')

    plt.tight_layout()
    plt.show()
else:
    print('Shadow classifier not available — skipping.')

## 7. Privacy vs utility bar chart

In [ ]:
labels = list(results.keys())
colors = [PALETTE.get(l, '#888888') for l in labels]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Privacy vs utility trade-off', fontsize=13)

# Attack AUC (lower = better privacy)
aucs = [results[l]['auc_thresh'] for l in labels]
bars1 = ax1.bar(range(len(labels)), aucs, color=colors, edgecolor='none', width=0.6)
ax1.axhline(0.5, color='black', lw=0.8, ls='--', alpha=0.4, label='Random baseline')
ax1.set_xticks(range(len(labels)))
ax1.set_xticklabels(labels, rotation=20, ha='right', fontsize=9)
ax1.set_ylim(0.4, 0.85)
ax1.set_ylabel('Threshold attack AUC')
ax1.set_title('Attack AUC (lower = better privacy)', fontsize=11)
ax1.legend(fontsize=8)
ax1.spines[['top','right']].set_visible(False)
for bar, val in zip(bars1, aucs):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.003,
             f'{val:.3f}', ha='center', va='bottom', fontsize=8)

# Holdout accuracy (higher = better utility)
accs = [results[l]['non_acc'] for l in labels]
bars2 = ax2.bar(range(len(labels)), accs, color=colors, edgecolor='none', width=0.6)
ax2.set_xticks(range(len(labels)))
ax2.set_xticklabels(labels, rotation=20, ha='right', fontsize=9)
ax2.set_ylim(0.0, 0.85)
ax2.set_ylabel('Holdout accuracy')
ax2.set_title('Model utility (higher = better)', fontsize=11)
ax2.spines[['top','right']].set_visible(False)
for bar, val in zip(bars2, accs):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 8. Signal distribution comparison

In [ ]:
signal = 'logit_gap'
n_configs = len(configs)
fig, axes = plt.subplots(1, n_configs, figsize=(4.5 * n_configs, 4))
if n_configs == 1:
    axes = [axes]
fig.suptitle(f'{signal} distribution — member vs non-member per defense', fontsize=13, y=1.03)

for ax, (label, df) in zip(axes, configs.items()):
    mem = df[df.split_name == 'member'][signal]
    non = df[df.split_name == 'nonmember'][signal]
    lo = np.percentile(pd.concat([mem, non]), 1)
    hi = np.percentile(pd.concat([mem, non]), 99)
    ax.hist(mem, bins=60, alpha=0.55, density=True, color='#3B8BD4',
            label='member', edgecolor='none', range=(lo, hi))
    ax.hist(non, bins=60, alpha=0.55, density=True, color='#D85A30',
            label='non-member', edgecolor='none', range=(lo, hi))
    ax.set_title(label, fontsize=11)
    ax.set_xlabel(signal)
    ax.legend(fontsize=8)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

## 9. Summary table (copy-paste ready)

In [ ]:
print(summary_df.to_string(index=False))